In [1]:
import numpy as np 
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from keras.models import Sequential
from keras.layers import Dense, Flatten, InputLayer
import keras

In [6]:
train = pd.read_csv('train.csv')
trainX, trainY = train.iloc[:, :train.shape[1]-1], train.iloc[:,train.shape[1]-1]

categoricals = trainX.loc[:, trainX.dtypes == 'O'].columns
len(categoricals)

43

In [8]:
cat_features = trainX.loc[:, categoricals]
cat_features = cat_features.apply(lambda x: x.fillna(x.mode()[0] if not x.mode().empty else 'None'))

In [9]:
# One hot encoding these features
ohe = OneHotEncoder(handle_unknown='ignore')
res = ohe.fit_transform(cat_features).toarray()

In [ ]:
cols = np.array([])
for i in range(cat_features.shape[1]):
    cols = np.concatenate((cols, categoricals[i] + '_' + np.sort(cat_features.iloc[:, i].unique())))    
cat = pd.DataFrame(res, columns=cols)

cat.shape 

(1460, 251)

In [11]:
# Dropping original categorical variables
trainX = trainX.drop(categoricals, axis=1)
# Concatenating the One Hot Encoded variables to the train dataset
trainX = pd.concat([trainX, cat], axis=1)

In [12]:
trainX.shape

(1460, 288)

In [13]:
trainX.fillna(trainX.median(), inplace=True)

In [16]:
# Normalizing training features and target
scalar_features = MinMaxScaler()
norm_train = scalar_features.fit_transform(trainX)
scalar_target = MinMaxScaler()
trainY = scalar_target.fit_transform(np.array(trainY).reshape(-1, 1))
# Defining the network
model = Sequential([
  Dense(norm_train.shape[1], input_dim=norm_train.shape[1], activation='sigmoid'),
  Dense(units=norm_train.shape[1]//2, activation='sigmoid'),
  Dense(units=1, activation='linear'), # Changed softmax to linear for regression
])
# Printing model summary
model.summary()


c:\Users\vigne\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 288)            │        83,232 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 144)            │        41,616 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │           145 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 124,993 (488.25 KB)

 Trainable params: 124,993 (488.25 KB)

 Non-trainable params: 0 (0.00 B)

In [17]:
model.compile(optimizer='sgd', loss='mean_squared_error')
model.fit(trainX, trainY, batch_size=512, epochs=20, verbose=1, validation_split=0.2)

Epoch 1/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 126ms/step - loss: 0.2617 - val_loss: 0.0145
Epoch 2/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - loss: 0.0143 - val_loss: 0.0122
Epoch 3/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - loss: 0.0115 - val_loss: 0.0114
Epoch 4/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - loss: 0.0107 - val_loss: 0.0112
Epoch 5/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - loss: 0.0103 - val_loss: 0.0107
Epoch 6/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - loss: 0.0097 - val_loss: 0.0105
Epoch 7/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - loss: 0.0093 - val_loss: 0.0105
Epoch 8/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - loss: 0.0090 - val_loss: 0.0102
Epoch 9/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - loss: 0.0089 - val_loss: 0.0100
Epoch 10/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - loss: 0.0083 - val_loss: 0.0095
Epoch 11/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - loss: 0.0083 - val_loss: 0.0097
Epoch 12/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - loss: 0.0082 - val_loss: 0.0104
